In [1]:
import torch
import torch.nn.functional as F
import requests

In [2]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()
print(words.__len__())

if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")


print(words[:10])
len(words)
block_size = 5
batch_size = words.__len__()
epochs = 2000
lr = 0.1

32033
Using device: mps
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [3]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [4]:
# encodes words indivdually
# def getXY(word):
#     X, Y = [], []
#     context = [0] * block_size
#     for ch in word + '.':
#         ix = stoi(ch)
#         X.append(context)
#         Y.append(ix)
#         context = context[1:] + [ix]
#         print(context)
#     return torch.tensor(X), torch.tensor(Y)

In [5]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [6]:
X, Y = build_dataset(words)
Xtr, Ytr = X[:30000], Y[:30000]
Xdev, Ydev = X[30000:], Y[30000:]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 5]) torch.Size([228146])
torch.Size([30000, 5]) torch.Size([30000])
torch.Size([198146, 5]) torch.Size([198146])


In [7]:
C = torch.randn((27, 2),requires_grad=True)
W1 = torch.randn((10, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn((100, 27), requires_grad=True)
b2 = torch.randn(27, requires_grad=True)
optimizer = torch.optim.AdamW([C, W1, b1, W2, b2], lr=lr)

In [9]:
for e in range(epochs):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]
    # forward
    emb = C[Xb] # (32, 3, 2)
    emb = emb.view(-1, 10) # (32*3, 10)
    h = torch.tanh(emb @ W1 + b1) # (32*3, 100)
    logits = h @ W2 + b2 # (32*3, 27)
    
    
    loss = F.cross_entropy(logits, Yb)
    # backward
    loss.backward()
    # update
    optimizer.step()
    optimizer.zero_grad()
    lr *= 0.999
    if e % 250 == 0:
        print(loss.item())

20.222036361694336
2.0979561805725098
2.0208611488342285
1.9062292575836182
2.6690897941589355
1.8617334365844727
1.9590877294540405
1.8333014249801636


In [18]:
cnt = 10
for i in range(cnt):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])] # (1, 3, 2)
        emb = emb.view(1, -1) # (1, 10)
        h = torch.tanh(emb @ W1 + b1) # (1, 100)
        logits = h @ W2 + b2 # (1, 27)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos(i) for i in out))

admie.
caylene.
chary.
emalis.
kaian.
lany.
anari.
cabrieth.
bexlyn.
roemi.
